# Audit Stage 1 answers on eligible training questions

Generate **one greedy answer per question** from the best Stage 1 checkpoint. This checks the errors the current model actually makes before building more Stage 2 pairs.

Use the **bioasq** Python kernel. Run cells in order. The default is the 0.5B gold-supported run; change `STAGE1_DIR` to a completed 3B Stage 1 directory to use that model. The base model, selected adapter, prompt template, and generation limits come from its manifest.

Eligibility requires a gold alias to occur within one `[BS]…[ES]` snippet. Matching ignores case and repeated whitespace but preserves punctuation and hyphens. This verifies textual occurrence, **not semantic correctness**. Official dev/test questions and the run's held-out curriculum questions are excluded.

Full resources are preserved. Over-budget prompts are recorded and skipped, never truncated. This is a training-data audit, not a replacement for dev evaluation.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_OFFLINE", "1")

import sys
import json
import gc
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_NAME = "Task-Structured Counterfactual Preference Mining"
ROOT = next((candidate for parent in [Path.cwd(), *Path.cwd().parents]
             for candidate in (parent, parent / PROJECT_NAME)
             if (candidate / "cse_dpo/audit_stage1_training_generations.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the BioASQ_MRES project.")
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cse_dpo.audit_stage1_training_generations import (
    resolve_stage1, prepare_questions, load_tokenizer_and_prompts,
    initialize_audit, write_json, write_csv, read_journal, export_audit,
    load_stage1_model, run_audit, score_completed_questions,
)
print("Project:", ROOT)
print("Python:", sys.executable)

## Settings

For a short check, set `LIMIT_QUESTIONS = 5`. Then set it to `None` and rerun the generation cell to finish all questions. Completed answers are saved after **every question** and reused on reruns.

Use a new `OUTPUT_DIR` if you change the model, prompt budget, or generation settings. The notebook refuses to mix incompatible results. Keep one notebook process writing to each output directory.

In [ ]:
STAGE1_DIR = ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_05b_c3_c1_curriculum_gold_supported_full_bp4096/stage_1_concept_learning"
CHECKPOINT_KEY = "best_mrr_adapter"  # Best official dev MRR, not the last checkpoint.
TRAIN_FILE = ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/train_prepared.json"
GOLD_FILE = ROOT / "data/training13b.json"  # All aliases; prepared output alone has only one.
EXCLUDE_CURRICULUM_HELDOUT = True

LOAD_IN_4BIT = True
MAX_PROMPT_TOKENS = None  # None: Stage 1 max_length; 0: no audit cap (model limit still checked).
MAX_NEW_TOKENS = None     # None: Stage 1 generated_eval_max_new_tokens.
LIMIT_QUESTIONS = None    # Set to 5 for a smoke run; None processes all eligible questions.
RETRY_FAILED = True      # Retry previously recorded CUDA OOMs, reuse completed answers.
RUN_OFFICIAL_SCORING = True

OUTPUT_DIR = ROOT / "Artifacts/cse_dpo/stage1_training_audits" / (STAGE1_DIR.parent.name + "_" + CHECKPOINT_KEY)
print("Audit output:", OUTPUT_DIR)

In [ ]:
stage = resolve_stage1(ROOT, STAGE1_DIR, CHECKPOINT_KEY)
questions, excluded, eligibility = prepare_questions(
    ROOT, stage, TRAIN_FILE, GOLD_FILE,
    exclude_curriculum_holdout=EXCLUDE_CURRICULUM_HELDOUT,
)
if not questions:
    raise ValueError("No eligible questions. Review the input paths and exclusion settings.")
print(json.dumps({
    "base_model": stage["base_model"], "selected_stage1_adapter": stage["adapter"],
    "eligible_questions": len(questions),
    "excluded_by_reason": eligibility["excluded_by_reason"],
}, indent=2))

## Prepare prompts and protect resume state

This cell loads only the tokenizer. It reports the full-prompt length distribution before loading model weights. Long questions remain in the audit with a skip status.

The prepared training file may have `05b` in its directory name; it supplies question/resource text. The tokenizer and inference model are taken from `STAGE1_DIR`.

In [ ]:
tokenizer, questions, prompt_bundle = load_tokenizer_and_prompts(ROOT, stage, questions)
resolved_prompt_budget = int(stage["config"].get("max_length", 4096) if MAX_PROMPT_TOKENS is None else MAX_PROMPT_TOKENS)
resolved_new_tokens = int(stage["config"].get("generated_eval_max_new_tokens", 64) if MAX_NEW_TOKENS is None else MAX_NEW_TOKENS)
if resolved_prompt_budget < 0 or resolved_new_tokens <= 0:
    raise ValueError("Prompt budget must be >= 0; generation allowance must be positive.")
settings = {
    "do_sample": False, "num_beams": 1, "num_return_sequences": 1,
    "load_in_4bit": LOAD_IN_4BIT, "max_prompt_tokens": resolved_prompt_budget,
    "max_new_tokens": resolved_new_tokens, "use_cache": False,
    "attn_implementation": stage["config"].get("attn_implementation", "sdpa"),
    "prompt_ref": stage["config"]["prompt_ref"], "truncate": False,
}
audit_config = initialize_audit(OUTPUT_DIR, stage, questions, settings, eligibility)
write_json(OUTPUT_DIR / "eligibility_summary.json", eligibility)
write_json(OUTPUT_DIR / "eligible_question_ids.json", [q["question_id"] for q in questions])
write_csv(OUTPUT_DIR / "excluded_questions.csv", excluded)
summary = export_audit(OUTPUT_DIR, questions, read_journal(OUTPUT_DIR / "generations.jsonl"))
lengths = pd.Series([q["prompt_tokens"] for q in questions], name="full_prompt_tokens")
display(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame())
print("Full prompts above audit budget:", sum(q["prompt_tokens"] > resolved_prompt_budget for q in questions) if resolved_prompt_budget else 0)
print("Resolved settings:", json.dumps(settings, indent=2))
print("Resume state:", summary["status_counts"])

## Generate answers

Run on a CUDA GPU. Each question uses the full prompt and greedy decoding with dropout disabled. OOM questions are logged; you can retry them after freeing GPU memory. An interruption preserves all finished questions.

Re-running the next two cells resumes this audit. There is no fine-tuning or LLM-judge call. For long prompts this audit's skip policy differs from evaluations that truncate inputs.

In [ ]:
import torch
# Release only this notebook's previously loaded audit model, if present.
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
model = load_stage1_model(stage, load_in_4bit=LOAD_IN_4BIT)
print("Loaded Stage 1 adapter:", stage["adapter"])
print("GPU:", torch.cuda.get_device_name())

In [ ]:
summary = run_audit(
    model, tokenizer, questions, OUTPUT_DIR,
    max_prompt_tokens=resolved_prompt_budget,
    max_new_tokens=resolved_new_tokens,
    limit_questions=LIMIT_QUESTIONS,
    retry_failed=RETRY_FAILED,
)
print(json.dumps(summary, indent=2))
print("Review:", OUTPUT_DIR / "manual_review.csv")

## Inspect the audit

- `all_questions_audit.csv`: every eligible question, prediction, aliases, snippet evidence, length, and status.
- `boundary_candidates.csv`: possible answers that are shorter/longer than a supported gold alias. Containment alone does not prove a boundary mistake.
- `manual_review.csv`: mismatches/format problems with editable `review_label`, `preferred_alias`, `approved_for_training`, and `review_notes` columns. These manual fields survive reruns.
- `skipped_or_failed.csv`: context-budget skips and OOMs.
- `generations.jsonl`: durable per-question records used for resume.

For manual labels, consider `too_short`, `too_long`, `partial_overlap`, `valid_alias`, `wrong_entity`, `unsupported`, `format_only`, or `ambiguous`. Compare the question and snippets, not just strings. An answer matching **any** gold alias is recognized, even if it differs from the prepared single output. Review a sample of apparently correct answers too: dataset gold can itself be imperfect.

No candidate is automatically approved as a C3 > C2 training pair. Generation-limit hits also need review.

In [ ]:
audit_df = pd.read_csv(OUTPUT_DIR / "all_questions_audit.csv")
display(audit_df.reindex(columns=["status", "category"]).groupby(["status", "category"], dropna=False).size().rename("questions").reset_index())
review_df = pd.read_csv(OUTPUT_DIR / "manual_review.csv", keep_default_na=False)
review_columns = [c for c in ["question_id", "question", "candidate", "gold_aliases", "category", "candidate_extractive", "boundary_relations", "snippets"] if c in review_df]
with pd.option_context("display.max_colwidth", 160):
    display(review_df[review_columns].head(20))
print("CSV files are in:", OUTPUT_DIR)

## Optional official BioASQ train metrics

This scores **completed generations only** using the project's Java evaluator and all gold aliases. Pending and skipped questions are excluded; always report coverage alongside MRR. This is a training diagnostic, not a dev score. A single valid prediction makes reciprocal rank either 0 or 1, so MRR equals accuracy in that case. Malformed multiple-answer outputs may be parsed differently; inspect the exported prediction file.

You can rerun this cell without generating again. Java must be available.

In [ ]:
if RUN_OFFICIAL_SCORING:
    records = read_journal(OUTPUT_DIR / "generations.jsonl")
    completed = sum(r["status"] == "complete" for r in records.values())
    print(f"Scoring coverage: {completed}/{len(questions)} eligible training questions")
    if completed:
        official_scores = score_completed_questions(ROOT, OUTPUT_DIR, questions)
        print(json.dumps(official_scores.get("aggregate", official_scores.get("overall", official_scores)), indent=2)[:6000])
        print("Official outputs:", OUTPUT_DIR / "official_bioasq")
    else:
        print("No completed predictions yet.")

## Release GPU memory

Run after generation; CSV inspection and official scoring do not need the model.

In [ ]:
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()